# Mundo 6 - API do IBGE

A API do IBGE é meio lixo supremo, mas da pra tirar algumas coisas interessantes.

### Conceitos importantes de hoje:
- APIs com endpoints diferentes
- Tratamento de dados

### 2 informações importantes obtidas pela API:
- Projeção populacional por país ou regiões
- Indicadores econômicos por país

### Documentação geral
https://servicodados.ibge.gov.br/api/docs

## Projeção populacional por país ou regiões

### Documentação:
https://servicodados.ibge.gov.br/api/docs/projecoes

### URL Base
https://servicodados.ibge.gov.br/api/v1/projecoes/populacao/{localidade}

Localidade deve ser:

* País: BR
* Regiões:
    * 1 - Norte
    * 2 - Nordeste
    * 3 - Sudeste
    * 4 - Sul
    * 5 - Centro-Oeste

In [18]:
import requests       # biblioteca pra fazer requisições HTTP (chamar APIs)
import pandas as pd   # biblioteca pra manipular dados em tabelas (DataFrames)
import json           # biblioteca pra trabalhar com dados no formato JSON

In [19]:
localidade = 'BR'     # define que vamos buscar dados do Brasil
url = f'https://servicodados.ibge.gov.br/api/v2/projecoes/populacao/{localidade}'
# monta a URL da API inserindo a variável localidade no link

response = requests.get(url)         # faz a requisição GET pra API (busca os dados)
data = json.loads(response.text)     # converte a resposta (texto) em dicionário Python
data                                 # exibe o resultado

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

## Indicadores econômicos por país

### Documentação:
https://servicodados.ibge.gov.br/api/docs/paises

### URL Base
https://servicodados.ibge.gov.br/api/v1/paises/{paises}/indicadores/{indicadores}

obs: Quando desejar mais de um indicador, ou país, utilizar o "|"

Localidade deve ser:

* País: Código de 2 digitos de acordo com a norma ISO 3166-1 ALPHA-2 (Exemplo: BR, US)

Indicador deve ser:

* Um número de acordo com o site do IBGE (ele define id's)

obs: Na documentação tem o id dos indicadores e o código de cada país

In [27]:
paises = 'BR|US'               # países que queremos consultar (| separa mais de um)
indicadores = '77823'          # código do indicador no IBGE (77823 = PIB per capita)
nome_pais = ['Brasil', 'EUA']  # lista com os nomes dos países, na mesma ordem do 'paises'

In [28]:
url = f'https://servicodados.ibge.gov.br/api/v1/paises/{paises}/indicadores/{indicadores}'
# monta a URL com os países e o indicador escolhido

response = requests.get(url)          # faz a requisição GET pra API
data = json.loads(response.text)      # converte a resposta em dicionário Python

lista_dfs = []    # cria uma lista vazia que vai guardar um DataFrame pra cada país

for i, nome_pais in enumerate(nome_pais):
# percorre a lista de países; i = índice numérico (0, 1...), nome_pais = nome do país

    lista_anos = []      # lista vazia pra guardar os anos
    lista_valores = []   # lista vazia pra guardar os valores do PIB

    for informacoes in data[0]['series'][i]['serie']:
    # acessa a série histórica do país i dentro do JSON retornado pela API
    # data[0] = primeiro indicador | ['series'][i] = série do país i | ['serie'] = dados ano a ano

        valores = list(informacoes.items())
        # converte cada item do dicionário (ano: valor) em uma lista de tuplas

        lista_anos.append(valores[0][0])     # pega o ano e adiciona na lista de anos
        lista_valores.append(valores[0][1])  # pega o valor e adiciona na lista de valores

    df = pd.DataFrame(list(zip(lista_anos, lista_valores)), columns=["Anos", f"{nome_pais}"]).dropna()
    # zip junta as duas listas em pares (ano, valor)
    # cria um DataFrame com colunas "Anos" e o nome do país
    # dropna() remove linhas com valores nulos

    lista_dfs.append(df)    # adiciona o DataFrame desse país na lista

df_final = pd.merge(lista_dfs[0], lista_dfs[1], on='Anos')
# junta os dois DataFrames (Brasil e EUA) pela coluna "Anos"

df_final = df_final.set_index("Anos")
# define a coluna "Anos" como índice da tabela

df_final = df_final.apply(pd.to_numeric, errors='coerce')
# converte todos os valores pra número; o que não conseguir vira NaN

print(df_final)    # exibe a tabela final

        Brasil       EUA
Anos                    
1990   3117.74  23888.60
1995   4756.75  28690.88
2000   3766.55  36329.97
2001   3176.29  37133.62
2002   2855.94  37997.74
2003   3090.61  39490.30
2004   3663.82  41724.64
2005   4827.78  44123.40
2006   5934.14  46301.99
2007   7409.69  48050.23
2008   8908.33  48570.06
2009   8678.66  47194.95
2010  11403.28  48650.66
2011  13396.62  50065.98
2012  12521.72  51784.41
2013  12458.89  53409.75
2014  12274.99  55304.32
2015   8936.20  57040.21
2016   8836.29  58206.61
2017  10080.51  60322.26
2018   9300.66  63201.05
2019   9029.83  65604.68
2020   7074.19  64411.37
2021   7972.54  71318.31
2022   9281.33  78035.18
2023  10294.87  82769.41


In [29]:
df_final['eua_maior'] = df_final ['EUA']/df_final['Brasil']

print (df_final)

        Brasil       EUA  eua_maior
Anos                               
1990   3117.74  23888.60   7.662153
1995   4756.75  28690.88   6.031614
2000   3766.55  36329.97   9.645424
2001   3176.29  37133.62  11.690878
2002   2855.94  37997.74  13.304810
2003   3090.61  39490.30  12.777510
2004   3663.82  41724.64  11.388289
2005   4827.78  44123.40   9.139480
2006   5934.14  46301.99   7.802645
2007   7409.69  48050.23   6.484783
2008   8908.33  48570.06   5.452207
2009   8678.66  47194.95   5.438046
2010  11403.28  48650.66   4.266374
2011  13396.62  50065.98   3.737210
2012  12521.72  51784.41   4.135567
2013  12458.89  53409.75   4.286879
2014  12274.99  55304.32   4.505447
2015   8936.20  57040.21   6.383050
2016   8836.29  58206.61   6.587223
2017  10080.51  60322.26   5.984048
2018   9300.66  63201.05   6.795330
2019   9029.83  65604.68   7.265328
2020   7074.19  64411.37   9.105123
2021   7972.54  71318.31   8.945494
2022   9281.33  78035.18   8.407758
2023  10294.87  82769.41   8

## Exercícios

- Exercício 139: Colete os dados de indivíduos com acesso a internet no BR da API de indicadores.
- Exercício 140: Colete os nomes mais registrados no Brasil desde 1950.
```

In [30]:
#gabarito 139

paises = 'BR'          # país que vamos consultar (apenas Brasil)
indicadores = '77857'  # código do indicador no IBGE (77857 = Pessoas com internet)

url = f'https://servicodados.ibge.gov.br/api/v1/paises/{paises}/indicadores/{indicadores}'
# monta a URL com o país e o indicador escolhido

response = requests.get(url)        # faz a requisição GET pra API
data = json.loads(response.text)    # converte a resposta em dicionário Python

lista_dfs = []      # lista vazia que vai guardar os DataFrames
lista_anos = []     # lista vazia pra guardar os anos
lista_valores = []  # lista vazia pra guardar os valores

for informacoes in data[0]['series'][0]['serie']:
# percorre a série histórica do Brasil
# data[0] = primeiro indicador | ['series'][0] = série do único país (BR) | ['serie'] = dados ano a ano
# no lugar desse "0", caso você puxe dois indicadores ao mesmo tempo, vai entrar em outro loop.

    valores = list(informacoes.items())
    # converte cada item do dicionário (ano: valor) em uma lista de tuplas

    lista_anos.append(valores[0][0])     # pega o ano e adiciona na lista de anos
    lista_valores.append(valores[0][1])  # pega o valor e adiciona na lista de valores

df = pd.DataFrame(list(zip(lista_anos, lista_valores)), columns=["Anos", "Brasil"]).dropna()
# zip junta as duas listas em pares (ano, valor)
# cria um DataFrame com colunas "Anos" e "Brasil"
# dropna() remove linhas com valores nulos

df = df.set_index("Anos")   # define a coluna "Anos" como índice da tabela

print(df)   # exibe a tabela

     Brasil
Anos       
1990      0
1995   0.11
2000   2.87
2001   4.53
2002   9.15
2003  13.21
2004  19.07
2005  21.02
2006  28.18
2007  30.88
2008  33.83
2009  39.22
2010  40.65
2011  45.69
2012  48.56
2013  51.04
2014  54.55
2015  58.33
2016  60.87
2017  67.47
2018  70.43
2019  73.91
2020  81.34
2021  80.69
2022  80.53
2023  84.15


In [31]:
#gabarito 140

url = 'https://servicodados.ibge.gov.br/api/v2/censos/nomes/ranking'
# URL da API de nomes do IBGE — retorna o ranking dos nomes mais registrados no Brasil desde 1950

response = requests.get(url)        # faz a requisição GET pra API
data = json.loads(response.text)    # converte a resposta em dicionário Python

lista_df = []   # lista vazia que vai guardar um DataFrame pra cada nome

for informacoes in data[0]['res']:
# data[0] = primeiro (e único) item do retorno da API
# ['res'] = lista com os resultados, cada item é um dicionário com nome, ranking e frequência

    df = pd.DataFrame(informacoes, index=[0])
    # cria um DataFrame com os dados de cada nome
    # index=[0] força o índice a começar em 0 (pois informacoes é um dicionário, não uma lista)

    lista_df.append(df)     # adiciona o DataFrame na lista

rankings_nomes = pd.concat(lista_df, ignore_index=True)
# concat junta todos os DataFrames da lista em um só
# ignore_index=True reinicia o índice do 0 ao final (evita índices duplicados)

rankings_nomes      # exibe a tabela com o ranking dos nomes

,nome,frequencia,ranking
0,MARIA,11734129,1
1,JOSE,5754529,2
2,ANA,3089858,3
3,JOAO,2984119,4
4,ANTONIO,2576348,5
5,FRANCISCO,1772197,6
6,CARLOS,1489191,7
7,PAULO,1423262,8
8,PEDRO,1219605,9
9,LUCAS,1127310,10
